In [7]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from scipy.stats import stats
import matplotlib.pyplot as plt

In [8]:
# 查看格式
df = pd.read_excel('./data/1、2、3.xlsx')

all_feature_id = np.array(df.columns[1:].tolist())
all_feature_name = np.array(df.iloc[0, 1:].values)
data_raw = df.iloc[1:, 1:].values.astype(np.float64)
classes = df.iloc[1:, 0].values.astype(int)

print(f"样本数: {data_raw.shape[0]}, 原始特征数: {data_raw.shape[1]}")
print(f"前5个特征ID: {all_feature_id[:5]}")
print(f"前5个代谢物名字: {all_feature_name[:5]}")
print(f"标签类别: {set(classes)}")

# 检查 Unknown 数量
unknown_count = np.sum(all_feature_name == 'Unknown')
print(f"保留的 Unknown 特征数: {unknown_count}")

样本数: 159, 原始特征数: 1734
前5个特征ID: [0 1 2 3 4]
前5个代谢物名字: ['Unknown' '2-Butenal, (Z)-' 'Unknown' 'Unknown'
 '4,5-Dihydrooxazole-5-one, 4-[4-acetoxy-3-methoxybenzylidene]-2-phenyl-']
标签类别: {np.int64(1), np.int64(2), np.int64(3)}
保留的 Unknown 特征数: 746


In [9]:
# ============================================================
# Step 1：丰度筛选
# ============================================================

mean_data = np.mean(data_raw, axis=0)

threshold_40 = np.percentile(mean_data, 40)

mask_step1 = mean_data > threshold_40

data_step1 = data_raw[:, mask_step1]

fids_step1_id = all_feature_id[mask_step1]
fids_step1_name = all_feature_name[mask_step1]

print("\n===== Step1 =====")
print("原始特征数:", data_raw.shape[1])
print("原始 Unknown 数:", np.sum(all_feature_name == 'Unknown'))
print("Step1 后特征数:", data_step1.shape[1])
print("Step1 后 Unknown 数:", np.sum(fids_step1_name == 'Unknown'))
print(f"阈值 P40 = {threshold_40:.4f}")


===== Step1 =====
原始特征数: 1734
原始 Unknown 数: 746
Step1 后特征数: 1040
Step1 后 Unknown 数: 434
阈值 P40 = 3947.8235


In [10]:
# ============================================================
# Step 2：IQR 筛选
# ============================================================

log_step1 = np.log1p(data_step1)

IQR = (
    np.percentile(log_step1, 75, axis=0)
    -
    np.percentile(log_step1, 25, axis=0)
)

threshold_iqr = np.percentile(IQR, 25)

mask_step2 = IQR >= threshold_iqr

data_step2 = log_step1[:, mask_step2]

fids_step2_id = fids_step1_id[mask_step2]
fids_step2_name = fids_step1_name[mask_step2]

IQR_ret = IQR[mask_step2]
IQR_rem = IQR[~mask_step2]

print("\n===== Step2 =====")
print("Step1 后特征数:", data_step1.shape[1])
print("Step2 后特征数:", data_step2.shape[1])
print("Step2 后 Unknown 数:", np.sum(fids_step2_name == 'Unknown'))
print(f"阈值 P25 = {threshold_iqr:.4f}")
print("Step2 去除特征数:", (~mask_step2).sum())


===== Step2 =====
Step1 后特征数: 1040
Step2 后特征数: 780
Step2 后 Unknown 数: 349
阈值 P25 = 1.3242
Step2 去除特征数: 260


In [11]:
col_names = [
    f"{fid}_{name}"
    for fid, name
    in zip(
        fids_step2_id,
        fids_step2_name
    )
]

df_filtered = pd.DataFrame(
    data_step2,
    columns=col_names
)

df_filtered.insert(
    0,
    'Class',
    classes
)

df_filtered.to_excel(
    'filtered_voc_step2.xlsx',
    index=False
)

print(
    f"筛选后数据："
    f"{df_filtered.shape[0]} 样本，"
    f"{df_filtered.shape[1] - 1} 特征"
)

print(
    "筛选后包含 Unknown 的特征数:",
    sum('Unknown' in str(x) for x in col_names)
)

筛选后数据：159 样本，780 特征
筛选后包含 Unknown 的特征数: 349


In [12]:
import pandas as pd
import torch

from torch.utils.data import Dataset
from scipy.io import savemat


class VOCDataset(Dataset):

    def __init__(
        self,
        path,
        label_map=None
    ):

        if label_map is None:
            label_map = {
                1: 0,
                2: 0,
                3: 1
            }

        df = pd.read_excel(path)

        self.y = torch.tensor(
            df['Class']
            .map(label_map)
            .values,
            dtype=torch.long
        )

        self.X = torch.tensor(
            df
            .drop(columns=['Class'])
            .values,
            dtype=torch.float32
        )

        self.feat_names = (
            df.columns[1:].tolist()
        )

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

    def to_mat(self, path):

        savemat(
            path,
            {
                'X':
                    self.X.numpy(),

                'y':
                    self.y.numpy()
                    .reshape(-1, 1),

                'feat_names':
                    self.feat_names,
            }
        )


ds = VOCDataset(
    'filtered_voc_step2.xlsx'
)

ds.to_mat(
    './data/voc_dataset_1+2_vs_3.mat'
)

print("=" * 60)

print(
    f"samples: "
    f"{len(ds)}"
)

print(
    f"features: "
    f"{len(ds.feat_names)}"
)

print(
    f"X shape: "
    f"{tuple(ds.X.shape)}"
)

print(
    f"y shape: "
    f"{tuple(ds.y.shape)}"
)

print(
    "class counts:",
    torch.bincount(ds.y).tolist()
)

print(
    "Unknown features:",
    sum(
        'Unknown' in str(name)
        for name in ds.feat_names
    )
)

print(
    "saved: "
    "./data/voc_dataset_1+2_vs_3.mat"
)

print("=" * 60)

samples: 159
features: 780
X shape: (159, 780)
y shape: (159,)
class counts: [106, 53]
Unknown features: 349
saved: ./data/voc_dataset_1+2_vs_3.mat


In [13]:
import scipy.io as sio

check_data = sio.loadmat(
    './data/voc_dataset_1+2_vs_3.mat'
)

print(
    "MAT X shape:",
    check_data['X'].shape
)

print(
    "MAT y shape:",
    check_data['y'].shape
)

print(
    "MAT feat_names:",
    len(
        check_data['feat_names'].flatten()
    )
)

MAT X shape: (159, 780)
MAT y shape: (159, 1)
MAT feat_names: 780
